In [7]:
# ==========================================================
# DOWNLOAD VIX + FED MACRO DATA
# ==========================================================

!pip install yfinance pandas_datareader

import yfinance as yf
import pandas as pd
import numpy as np
from pandas_datareader import data as pdr
from google.colab import files

START = "2020-01-01"
END = "2026-01-01"

# ==========================================================
# PART 1 — VIX
# ==========================================================
print("Downloading VIX...")

vix = yf.download("^VIX", start=START, end=END, auto_adjust=False)

# Fix MultiIndex columns nếu có
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)

vix = vix.reset_index()

print("VIX columns:", vix.columns.tolist())

# Drop volume
if "Volume" in vix.columns:
    vix = vix.drop(columns=["Volume"])

# Dùng Adj Close nếu có, không thì dùng Close
price_col = "Adj Close" if "Adj Close" in vix.columns else "Close"

vix["VIX_return"] = np.log(
    vix[price_col] / vix[price_col].shift(1)
)

vix["VIX_volatility_5"] = (
    vix["VIX_return"]
    .rolling(5)
    .std()
)

vix_final = vix[
    ["Date", "Close", "VIX_return", "VIX_volatility_5"]
]

print("VIX shape:", vix_final.shape)

# ==========================================================
# PART 2 — FRED
# ==========================================================
print("Downloading FRED data...")

effr = pdr.DataReader("EFFR", "fred", START, END)
dgs2 = pdr.DataReader("DGS2", "fred", START, END)
dgs10 = pdr.DataReader("DGS10", "fred", START, END)

macro = pd.concat([effr, dgs2, dgs10], axis=1)

# Weekend / holiday fill
macro = macro.ffill()

macro = macro.reset_index()

print("Macro columns:", macro.columns.tolist())

# Fix date column
date_col = macro.columns[0]
macro = macro.rename(columns={date_col: "Date"})

# Feature engineering
macro["EFFR_change"] = macro["EFFR"].diff()
macro["Yield_Spread"] = macro["DGS10"] - macro["DGS2"]

macro_final = macro[
    [
        "Date",
        "EFFR",
        "EFFR_change",
        "DGS2",
        "DGS10",
        "Yield_Spread"
    ]
]

print("Macro shape:", macro_final.shape)

# ==========================================================
# EXPORT
# ==========================================================
vix_final.to_csv("VIX_features.csv", index=False)
macro_final.to_csv("Macro_features.csv", index=False)

files.download("VIX_features.csv")
files.download("Macro_features.csv")

[*********************100%***********************]  1 of 1 completed

VIX columns: ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
VIX shape: (1508, 4)


Macro columns: ['DATE', 'EFFR', 'DGS2', 'DGS10']
Macro shape: (1567, 6)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>